# Code to generate an STL file of an oblate spheroidal hailstone with roughly equidistant lobes.
Produces two halves that screw together, with an interior space for instrumentation.
Rotation is carefully structured so the halves finish screwing together at the exact matching point.

In [1]:
import numpy as np
from matplotlib import cm
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pyvista as pv
import trimesh
from copy import deepcopy
import pymeshfix as mf
import cadquery as cq
from cq_warehouse.thread import PlasticBottleThread
from geographiclib.geodesic import Geodesic
%matplotlib widget

## Set oblate spheroidal and lobe properties as desired

In [2]:
#Number of lobes you want on your hailstone.
n_lobes = 150

#How narrow you want your lobes to be. Decimal number between
# 0 (lobe nonexistent, no point!). to 0.45 (as large as possible 
# without overlap, if placed symmetrically/no random perturbatios)
# to something higher (at 1.0, they will fully overlap with the
# neighboring node if placed symmetrically). Note lobe width is also 
# dependent on the number of lobes, and the sine/cosine equation 
# used to generate them. That equation would need to be modified 
# in code below.
lobe_factor = 1.0

#Lobe height factor. Setting it to 1 will make the lobes a factor
# of 1.5*radius. Setting it to 0 will make the lobes the same
# level as the radius (non-existent).
height_factor = 0.5

#Radii of the oblate spheroid underlying the lobes.
radius_a = 40. #mm, major/maximum dimension, assumed to be horizontal x dimension
radius_c = radius_a #other horizontal axis (y). They are assumed to be equal here.
radius_b = 30. #mm minor/minimum dimension, assumed to be vertical z dimension.

#create a geodesic instance with these dimensions
f = 1 - radius_b / radius_a #"flattening" of the spheroid
geod = Geodesic(radius_a, f)

#How large a random factor is used to perturb lobes away from equal spacing.
#If set to 1, d_phi and d_lambda angle distances from one lobe center to the 
# next are multiplied by a random number chosen from a range of [-0.5, 0.5).
#If set to 0.5, angle distances are multipled by a random number b/w [-0.25, 0.25). 
#Etc. Recommend b/w 0.25 and 0.5.
rand_factor = 0.25

#Use pyvista to generate an oblate spheroid. Increasing u_res, v_res, w_res will increase file size.
#https://docs.pyvista.org/api/utilities/_autosummary/pyvista.parametricellipsoid
ellipsoid_surf = pv.ParametricEllipsoid(xradius = radius_a, yradius=radius_c, zradius=radius_b)

### Define a function to fix the normals of both the sphere and lobes (courtesy of Joshua Soderholm)

In [3]:
def recalculate_normals_from_centroid(mesh):
    """
    Recalculate all normals to point outward from the mesh centroid.
    Handles triangular, quad, and mixed polygon meshes.
    
    Parameters
    ----------
    mesh : pv.PolyData
        The mesh to process
        
    Returns
    -------
    mesh : pv.PolyData
        The mesh with corrected normals
    """
    # Get the centroid of the entire mesh
    centroid = mesh.center
    
    # Get cell centers
    cell_centers = mesh.cell_centers().points
    
    # Calculate vectors from centroid to each cell center
    outward_vectors = cell_centers - centroid
    
    # Normalize these vectors to get desired normal directions
    desired_normals = outward_vectors / np.linalg.norm(outward_vectors, axis=1, keepdims=True)
    
    # Compute the current normals
    mesh.compute_normals(cell_normals=True, point_normals=False, inplace=True)
    current_normals = mesh.cell_data['Normals']
    
    # Check which normals need flipping
    dot_products = np.sum(current_normals * desired_normals, axis=1)
    needs_flipping = dot_products < 0
    
    # Process faces - need to handle variable polygon sizes
    faces = mesh.faces.copy()
    n_cells = mesh.n_cells
    
    # Parse faces array and flip as needed
    idx = 0
    for cell_id in range(n_cells):
        # First value is number of points in this face
        n_points = faces[idx]
        
        # Get the vertex indices for this face
        vertex_indices = faces[idx+1:idx+1+n_points]
        
        # If this face needs flipping, reverse the vertex order
        if needs_flipping[cell_id]:
            faces[idx+1:idx+1+n_points] = vertex_indices[::-1]
        
        # Move to next face
        idx += n_points + 1
    
    # Update the mesh faces
    mesh.faces = faces
    
    # Recompute normals after flipping
    mesh.compute_normals(cell_normals=True, point_normals=True, consistent_normals=True)
    
    print(f"Flipped {needs_flipping.sum()} out of {len(needs_flipping)} faces")
    
    return mesh

### Function to fix the small holes that make the shape not watertight

In [4]:
def fill_all_hole_loops(mesh, holes):
    """
    Fill all hole loops on a triangulated PyVista mesh by creating fan triangles
    from each loop's centroid to its boundary vertices.
    
    Parameters
    ----------
    mesh : pyvista.PolyData
        Triangulated input mesh (will not be modified in-place).
    holes : pyvista.PolyData
        Lines polydata returned by MeshFix.extract_holes() (edge format).
    
    Returns
    -------
    pyvista.PolyData
        New mesh with all holes filled (triangles appended).
    """
    import numpy as np
    import pyvista as pv
    
    if holes is None or holes.n_points == 0 or holes.n_cells == 0:
        return mesh.copy()
    
    # Parse holes.lines to extract edges (2-point line segments)
    lines = holes.lines.astype(int)
    edges = []
    idx = 0
    while idx < len(lines):
        n = int(lines[idx])
        if n == 2:  # edge
            p0, p1 = lines[idx+1], lines[idx+2]
            edges.append((p0, p1))
        idx += n + 1
    
    if not edges:
        return mesh.copy()
    
    # Reconstruct closed loops from edges
    # Build adjacency: for each point, what points does it connect to?
    from collections import defaultdict
    adj = defaultdict(list)
    for p0, p1 in edges:
        adj[p0].append(p1)
        adj[p1].append(p0)
    
    # Extract loops by following edges
    loops = []
    visited_edges = set()
    
    for start_p, neighbors in adj.items():
        for next_p in neighbors:
            edge_key = (min(start_p, next_p), max(start_p, next_p))
            if edge_key in visited_edges:
                continue
            
            # Trace loop starting from this edge
            loop = [start_p, next_p]
            current = next_p
            prev = start_p
            visited_edges.add(edge_key)
            
            while current != start_p:
                # Find next neighbor (not the one we came from)
                next_neighbors = [p for p in adj[current] if p != prev]
                if not next_neighbors:
                    break  # Dead end, not a closed loop
                
                next_p = next_neighbors[0]
                edge_key = (min(current, next_p), max(current, next_p))
                visited_edges.add(edge_key)
                
                if next_p == start_p:
                    # Loop closed
                    if len(loop) >= 3:
                        loops.append(np.array(loop))
                    break
                else:
                    loop.append(next_p)
                    prev = current
                    current = next_p
    
    if not loops:
        print("No closed loops found in hole edges")
        return mesh.copy()
    
    print(f"Found {len(loops)} hole loops")
    
    # Copy original mesh data
    orig_pts = mesh.points.copy()
    orig_faces = mesh.faces.astype(int)
    
    # Convert existing faces to explicit triangle list. Not necessary if already triangulated.
    tris = []
    j = 0
    while j < len(orig_faces):
        nv = int(orig_faces[j])
        verts = orig_faces[j + 1: j + 1 + nv].tolist()
        if nv == 3:
            tris.append(verts)
        else:
            # fan-triangulate polygon
            for k in range(1, nv - 1):
                tris.append([verts[0], verts[k], verts[k + 1]])
        j += nv + 1
    
    tris = np.array(tris, dtype=int)
    
    # Start with original mesh points, then append hole boundary points and centroids
    new_points = orig_pts.tolist()
    new_tris = tris.tolist()
    
    # Map from holes.points indices to new_points indices
    hole_pts_offset = len(new_points)
    for pt in holes.points:
        new_points.append(pt.tolist())
    
    # Fill each hole loop with fan triangles to its centroid
    for loop_idx, loop in enumerate(loops):
        # loop contains indices into holes.points
        # Map them to indices in new_points
        loop_indices_in_new = [hole_pts_offset + idx for idx in loop]
        
        # Compute centroid from actual hole boundary points
        coords = holes.points[loop]
        centroid = coords.mean(axis=0)
        centroid_idx = len(new_points)
        new_points.append(centroid.tolist())
        
        # Create fan triangles: (loop[i], loop[i+1], centroid) for each edge
        L = len(loop_indices_in_new)
        
        for k in range(L):
            i0 = loop_indices_in_new[k]
            i1 = loop_indices_in_new[(k + 1) % L]
            new_tris.append([i0, i1, centroid_idx])
    
    # Build VTK faces array from triangles: [3, i0, i1, i2, 3, j0, j1, j2, ...]
    new_tris = np.array(new_tris, dtype=np.int64)
    
    # Create faces array: prepend count (3) to each triangle
    faces_list = []
    for tri in new_tris:
        faces_list.append(3)
        faces_list.extend(tri)
    
    faces_vtk = np.array(faces_list, dtype=np.int64)
    
    # Create and return new mesh
    new_pts_arr = np.array(new_points, dtype=float)
    print(f"Created mesh with {len(new_tris)} triangles, {len(new_pts_arr)} points")
    new_mesh = pv.PolyData(new_pts_arr, faces_vtk)
    
    # Recompute normals 
    new_mesh.compute_normals(cell_normals=True, point_normals=True, consistent_normals=True)
    
    return new_mesh

## Add equally spaced lobes to the oblate spheroidal surface

In [5]:
from scipy.integrate import quad
from scipy.optimize import brentq

# Create a range of equidistant points on an oblate spheroidal surface. 
# This follows the same framework as for a sphere via Deserno 2004, but
# is modified to account for non-sphericity.

# A few differences from the spherical notebook. lambda is the longitudinal angle (0 to 2pi). 
# phi is the geodetic latitude - not the same as geocentric latitude. 
# It runs from -pi/2 to pi/2. 
# Recommended reading:
#  https://www.fatiando.org/harmonica/v0.7.0/user_guide/coordinate_systems.html
#  https://jerrymahun.com/index.php/home/open-access/93-updating/428-chapter-c-ellipsoid


#create a random number generator
rng = np.random.default_rng(42) #will generate b/w 0 and 1

lobe_centers = [] #just the center points in cartesian space
lobes = [] #successive sets of points, including circles around center points, one for each lobe
lobe_count = 0 #actual number of lobes
lobe_lambdas = []
lobe_phis = []
lobe_rpvs = [] #radius of "prime vertical", line from z-axis to the point normal to surface of 
               #spheroid at point phi, lambda. Angle between z-axis and the rpv is phi.

#get the coordinates of all lobe centers

#determine non-lobey shape surface area
e = (1-(radius_b**2./radius_a**2))**0.5 #eccentricity
sa = 2*np.pi*radius_a**2 + np.pi*(radius_b**2/e)*np.log((1+e)/(1-e))

#find average distance (d) between points.
a = sa / n_lobes
d = np.sqrt(a)

#calculate number of points in N/S direction, along a meridian (pole-pole line at constant lambda).
#ellipsoid circumference along half of the meridian, using an approximation:
c_meridian = (np.pi/2) * (3*(radius_a+radius_b) - ((3*radius_a+radius_b)*(radius_a+3*radius_b))**0.5)
#given average distance b/w points, get number of points 
M_phi = np.round(c_meridian/d) 
#and now distance between the points along the merdian
d_phi = c_meridian / M_phi

#given that distance, what is our distance between points along the constant latitude circle?
d_lambda = a / d_phi
print ('d_phi, d_lambda: ', d_phi, d_lambda)

# --- NEW: meridian arc length functions for accurate latitude placement ---
def meridian_integrand(phi, radius_a, e):
    """Exact arc length element ds/dphi along the meridian at geodetic latitude phi."""
    return radius_a * (1 - e**2) / (1 - e**2 * np.sin(phi)**2)**1.5

def arc_from_south_pole(phi_target):
    """Arc length from south pole (-pi/2) to geodetic latitude phi_target."""
    arc, _ = quad(meridian_integrand, -np.pi/2, phi_target, args=(radius_a, e))
    return arc

def phi_from_arc(s_target):
    """Find geodetic latitude phi such that arc length from south pole equals s_target."""
    if s_target <= 0:
        return -np.pi/2
    if s_target >= c_meridian:
        return np.pi/2
    f = lambda phi: arc_from_south_pole(phi) - s_target
    return brentq(f, -np.pi/2, np.pi/2, xtol=1e-10)

#for each point along the half-meridian, generate a circle of constant geodetic latitude
for m in np.arange(0,M_phi):

    #First, we generate the equally space latitudes in projected spherical space (aka authalic latitudes or beta)
    #generate a random number for starting phi, so we don't always start at 0 or d_phi
    rand = rand_factor*(rng.random()-0.5) 
    #OLD
    # #generate beta at equally spaced areas
    # beta = 0.95*np.pi*((m+0.5+rand) / M_phi - 0.5)
    # #convert authalic latitudes to geodetic, via approximation
    # phi = beta + (e**2/3)*np.sin(2*beta) + (3*e**4/20)*np.sin(4*beta) + (5*e**6/56)*np.sin(6*beta)
    # NEW: place latitude circles by inverting the meridian arc integral directly
    s_target = np.clip((m + 0.5 + rand) * d_phi, 0.01 * d_phi, c_meridian - 0.01 * d_phi)
    phi = phi_from_arc(s_target)
        
    #Calculate circumference of horizontal cross-section at this latitude, phi.
    #Radius in the prime vertical (line normal to the spheroidal surface at this point)
    r_pv = radius_a / ((1 - e**2*(np.sin(phi))**2)**0.5)
    #Circumference of the horizontal circle cross-section with that radius, scaled for latitude
    c_parallel = 2*np.pi * r_pv * np.cos(phi)
    
    #number of equidistant points along this circle
    M_lambda = np.round(c_parallel/d_lambda)
    
    
    #for each point along this latitude circle, generate M_lambda equidistant points
    for n in np.arange(0,M_lambda):
        #a random number for starting lambda
        rand = rand_factor**(rng.random()-0.5) 
        #equally spaced lambdas(+/- rand factor)
        lambda_angle = 2*np.pi*(n+rand)/M_lambda
        
        #add a small random factor to phi, so lobes will be slightly
        #displaced off the latitude circle of constant phi
        #first, convert d_phi, distance between latitude circles, to latitude instead of arclength
        d_phi_angle = np.pi * (d_phi / c_meridian)
        #now add random factor
        rand = rand_factor*(rng.random()-0.5) 
        new_phi = phi + d_phi_angle*rand
        new_phi = np.clip(new_phi, -np.pi/2, np.pi/2)
        
        #calculate new radius in prime vertical with this new phi
        new_r_pv = radius_a / ((1 - e**2*(np.sin(new_phi))**2)**0.5)
        
        #convert to cartesian coordinates and store
        x = new_r_pv * np.cos(new_phi)*np.cos(lambda_angle)
        y = new_r_pv * np.cos(new_phi)*np.sin(lambda_angle)
        z = new_r_pv * (1-e**2) * np.sin(new_phi)
        lobe_lambdas.append(lambda_angle)
        lobe_phis.append(new_phi)
        lobe_rpvs.append(new_r_pv)
        lobe_centers.append((x,y,z))
        lobe_count += 1

#Check to make sure we didn't leave a gap at either pole

#north pole first (phi = -pi/2)
if arc_from_south_pole(min(lobe_phis)) > d_phi:
    #add a new lobe, a random offset from the north pole
    rand = (rand_factor/2.)*rng.random() #will be between 0 and 0.125
    phi = phi_from_arc(d_phi * rand)
    rand = rng.random() #between 0 and 1
    lambda_angle = 2*np.pi*rand #random longitude b/w 0 and 2 pi
    #given this new latitude, determine new r_pv
    r_pv = radius_a / ((1 - e**2*(np.sin(phi))**2)**0.5)

    #calculate cartesian parameters and store    
    lobe_lambdas.append(lambda_angle)
    lobe_phis.append(phi)
    lobe_rpvs.append(r_pv)
    x = r_pv * np.cos(phi)*np.cos(lambda_angle)
    y = r_pv * np.cos(phi)*np.sin(lambda_angle)
    z = r_pv * (1-e**2) * np.sin(phi)
    lobe_centers.append((x,y,z))
    lobe_count += 1    
    
#and check again for the south pole (phi = pi/2)
if c_meridian - arc_from_south_pole(max(lobe_phis)) > d_phi:
    #add a new lobe, a random offset from the south pole
    rand = (rand_factor/2.)*rng.random() #will be between 0 and 0.125
    phi = phi_from_arc(c_meridian - d_phi * rand)
    rand = rng.random() #between 0 and 1
    lambda_angle= 2*np.pi*rand #random longitude b/w 0 and 2pi
    #given this new latitude, determine new r_pv
    r_pv = radius_a / ((1 - e**2*(np.sin(phi))**2)**0.5)
    
    #calculate cartesian parameters and store    
    lobe_lambdas.append(lambda_angle)
    lobe_phis.append(phi)
    lobe_rpvs.append(r_pv)
    x = r_pv * np.cos(phi)*np.cos(lambda_angle)
    y = r_pv * np.cos(phi)*np.sin(lambda_angle)
    z = r_pv * (1-e**2) * np.sin(phi)
    lobe_centers.append((x,y,z))
    lobe_count += 1    


# # Add this diagnostic after generating lobe_centers but before the lobe circles

lobe_centers_np = np.array(lobe_centers)
print("=== LOBE CENTERS AXIS RANGES ===")
print(f"x range: [{lobe_centers_np[:,0].min():.2f}, {lobe_centers_np[:,0].max():.2f}]")
print(f"y range: [{lobe_centers_np[:,1].min():.2f}, {lobe_centers_np[:,1].max():.2f}]")
print(f"z range: [{lobe_centers_np[:,2].min():.2f}, {lobe_centers_np[:,2].max():.2f}]")

print("\n=== EXPECTED ===")
print(f"x radius: {radius_a}")
print(f"y radius: {radius_c}")
print(f"z radius: {radius_b}")


d_phi, d_lambda:  11.05174589545837 10.166090984079078
=== LOBE CENTERS AXIS RANGES ===
x range: [-38.81, 38.86]
y range: [-39.01, 39.05]
z range: [-29.80, 29.76]

=== EXPECTED ===
x radius: 40.0
y radius: 40.0
z radius: 30.0


## Given central points, now generate the lobes around each one

In [6]:
#convert d_lambda to angle between longitude points instead of distance
c_azimuth = 2*np.pi*radius_a #azimuthal (e-w) cross-section is just a circle since radius_a=radius_c
d_lambda_angle = np.pi * (d_lambda / c_azimuth)

#reference radius for scaling purposes - equal to 1 at halfway b/w a and b
r_ref = (radius_a + radius_b) / 2. 

#now calculate "circles" around each lobe point, starting at a central angle of alpha 
# from each lobe point
alpha_start = lobe_factor*np.minimum(d_phi_angle, d_lambda_angle)
num_t_at_start = 200 #originally 40, but that looked a little choppy when printed
circ_at_start = 2*np.pi*alpha_start
dt = circ_at_start / num_t_at_start
#lobe_heights = radius + np.sin(np.pi/2 * np.linspace(0,0.5,10)) #equation too sharp
#The below uses https://easings.net/#easeInOutSine instead
lobe_heights = 1+height_factor*\
               (-(np.cos(np.pi*np.linspace(0,1.0,num_t_at_start))-1)/4)   
#lobe_heights = np.ones(num_t_at_start) #for debugging

for phi, lambda_angle, lobe_center, r_pv_center in zip(lobe_phis, lobe_lambdas, lobe_centers, lobe_rpvs):
    lobe_points = []
    
    #cycle through successively smaller alphas, setting the radius/height increasingly higher
    # each time
    alphas = np.linspace(alpha_start, 0, num_t_at_start)
                    
    xs, ys, zs = [], [], []   #3d cartesian points for lobes               
                
    for alpha, lobe_height in zip(alphas[:-1], lobe_heights[:-1]):
    
        #calculate number of t needed for this alpha
        circ_here = 2*np.pi*alpha
        num_t = int(np.round(circ_here/dt))
        t = np.linspace(0,2*np.pi,num_t)
        
        #find the distance between center point (phi, lambda) and point alpha angle away.
        #to do that, calculate the radius of curvature in both the prime vertical and meridional
        # directions. 
        #we already know r_pv at this point, it is r_pv_center
        # meridional radius is from https://jerrymahun.com/index.php/home/open-access/93-updating/428-chapter-c-ellipsoid?start=3
        # EXCEPT! That website apparently has an error! See https://www.oc.nps.edu/oc2902w/geodesy/radiigeo.pdf,
        # which uses 1.5 as the denominator exponent instead of 0.5!
        r_m = (radius_a*(1-e**2.))/ (1 - e**2.*np.sin(phi)**2)**1.5
        #distance = (r_ref / r_m)* r_pv_center * alpha
        distance = r_pv_center * alpha


        #determine lat,lon of all the points in the circle
        for azimuth in t: #all the angles/headings about the circle
        
            #the function below can't handle latitude or longitude outside of the -90,90 and 0,360 range
            if lambda_angle < 0:
                lambda_angle = 2*np.pi + lambda_angle
            # elif lambda_angle > 2*np.pi:
            #     lambda_angle = lambda_angle - 2*np.pi
            
            # if phi > np.pi/2:
            #     phi = -np.pi/2 - (phi + np.pi/2)
            
            # Use geographiclib to find point at this distance and azimuth
            #print (np.degrees(phi), np.degrees(lambda_angle), np.degrees(azimuth), distance)
            result = geod.Direct(np.degrees(phi), np.degrees(lambda_angle), np.degrees(azimuth), distance)
            lat, lon = result['lat2'], result['lon2']
            #print (lat, lon)
            
            # Convert back to radians
            lat_rad = np.radians(lat)
            lon_rad = np.radians(lon)
            
            # Calculate r_pv at this new circle point location
            r_pv_point = radius_a / ((1 - e**2*(np.sin(lat_rad))**2)**0.5)
            
            # #get the cartesian coordinates for these points on the ellispoid surface
            x_base = r_pv_point * np.cos(lat_rad) * np.cos(lon_rad)
            y_base = r_pv_point * np.cos(lat_rad) * np.sin(lon_rad)
            z_base = r_pv_point * (1-e**2) * np.sin(lat_rad)

            # Compute surface normal at this point (outward normal for ellipsoid)
            # For an ellipsoid, the normal is proportional to (x/a^2, y/b^2, z/c^2)
            nx = x_base / (radius_a**2)
            ny = y_base / (radius_c**2)
            nz = z_base / (radius_b**2)
            norm_len = np.sqrt(nx**2 + ny**2 + nz**2)
            nx /= norm_len
            ny /= norm_len
            nz /= norm_len
            
            # Project outward along normal direction instead of radial scaling
            projection_distance = (lobe_height - 1.0) * radius_b #r_pv_point
            x = x_base + nx * projection_distance
            y = y_base + ny * projection_distance
            z = z_base + nz * projection_distance
            
            #now store these
            xs.append(x)
            ys.append(y)
            zs.append(z)
            lobe_points.append((x,y,z))


    #last point is right at alpha = 0
    alpha = alphas[-1]
    lobe_height = lobe_heights[-1]
    
    # # For the final center point, also use normal projection
    x_base = r_pv_center * np.cos(phi) * np.cos(lambda_angle)
    y_base = r_pv_center * np.cos(phi) * np.sin(lambda_angle)
    z_base = r_pv_center * (1-e**2) * np.sin(phi)
    
    # Compute surface normal at lobe center
    nx = x_base / (radius_a**2)
    ny = y_base / (radius_c**2)
    nz = z_base / (radius_b**2)
    norm_len = np.sqrt(nx**2 + ny**2 + nz**2)
    nx /= norm_len
    ny /= norm_len
    nz /= norm_len
    
    # Project outward along normal
    projection_distance = (lobe_height - 1.0) * radius_b# r_pv_center
    x = x_base + nx * projection_distance
    y = y_base + ny * projection_distance
    z = z_base + nz * projection_distance

    #now store these
    xs.append(x)
    ys.append(y)
    zs.append(z)
    lobe_points.append((x,y,z))
    #print (alpha, ': ', len(xs), len(ys), len(zs), len(lobe_points))
   
    lobes.append(lobe_points)


lobes_np = np.array(lobes)
print ('')
print("=== LOBE POINTS AXIS RANGES ===")
print(f"x range: [{lobes_np[:,:,0].min():.2f}, {lobes_np[:,:,0].max():.2f}]")
print(f"y range: [{lobes_np[:,:,1].min():.2f}, {lobes_np[:,:,1].max():.2f}]")
print(f"z range: [{lobes_np[:,:,2].min():.2f}, {lobes_np[:,:,2].max():.2f}]")

print ('len of lobes', len(lobes))
print ('len of lobe centers', len(lobe_centers))


=== LOBE POINTS AXIS RANGES ===
x range: [-46.02, 46.05]
y range: [-46.24, 46.29]
z range: [-37.28, 37.23]
len of lobes 151
len of lobe centers 151


## Reconstruct a surface by merging lobe points in the point cloud to our oblate spheroid

In [ ]:
#calculate a separate surface for each lobe
lobe_surfs = []
for lobe in lobes:
    lobe_points = np.array(lobe)
    #create a point cloud from each set of lobe points
    lobe_point_cloud = pv.PolyData(lobe_points)
    lobe_surf = lobe_point_cloud.reconstruct_surface()
    #triangulate our lobes instead - fast and good for testing, but not good for final product
    #lobe_surf = lobe_point_cloud.delaunay_2d()
    lobe_surfs.append(lobe_surf)
print('done with lobe for loop')

#add our surfaces together
total_surf = deepcopy(ellipsoid_surf)
for lobe_surf in lobe_surfs:
    #total_surf = total_surf + lobe_surf 
    #below gives fewer duplicate points to clean
    total_surf = total_surf.merge(lobe_surf)

#convert polygons to triangles
total_surf = total_surf.triangulate()    
# Fix the surface normals to make them all point outward
#cell_normals_recal = recalculate_normals_from_centroid(total_surf)

# #and plot, by individual lobe and then combined surface    
pv.global_theme.color_cycler = 'default'
pl = pv.Plotter(shape=(1, 2))
pl.subplot(0, 0)
_ = pl.add_mesh(ellipsoid_surf, opacity=0.5)
for lobe_surf in lobe_surfs:
    _ = pl.add_mesh(lobe_surf)
    
pl.subplot(0, 1)
_ = pl.add_mesh(total_surf,color='blue')
pl.show()



done with lobe for loop


Widget(value='<iframe src="http://localhost:55974/index.html?ui=P_0x30370f710_0&reconnect=auto" class="pyvista…

Exception raised
KeyError('015145f1c7d0777bc36edb8c2017ca0f_77616L')
Traceback (most recent call last):
  File "/Users/rselin/miniconda3/envs/3dprint/lib/python3.11/site-packages/wslink/protocol.py", line 317, in onCompleteMessage
    results = func(*args, **kwargs)
              ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/rselin/miniconda3/envs/3dprint/lib/python3.11/site-packages/trame_vtk/modules/vtk/protocols/local_rendering.py", line 33, in get_array
    self.context.get_cached_data_array(data_hash, binary)
  File "/Users/rselin/miniconda3/envs/3dprint/lib/python3.11/site-packages/trame_vtk/modules/vtk/serializers/synchronization_context.py", line 47, in get_cached_data_array
    cache_obj = self.data_array_cache[p_md5]
                ~~~~~~~~~~~~~~~~~~~~~^^^^^^^
KeyError: '015145f1c7d0777bc36edb8c2017ca0f_77616L'

Exception raised
KeyError('43b408c84c8a77997e035a061fe113bc_19216L')
Traceback (most recent call last):
  File "/Users/rselin/miniconda3/envs/3dprint/lib/python3.11/site-pack

## Now fix any holes

In [9]:
tri_surf = total_surf.triangulate()
mesh_to_fix = mf.MeshFix(tri_surf)
holes = mesh_to_fix.extract_holes()

# Show holes before filling
p = pv.Plotter()
p.add_mesh(tri_surf, color='blue')
p.add_mesh(holes, color='red', line_width=8)
p.show()

# Fill all holes
filled = fill_all_hole_loops(tri_surf, holes)
print(f"Original mesh: {tri_surf.n_points} points, {tri_surf.n_cells} cells")
print(f"Filled mesh: {filled.n_points} points, {filled.n_cells} cells")

# Check if watertight (convert to trimesh to access is_watertight)
filled_trimesh = trimesh.Trimesh(vertices=filled.points, faces=filled.faces.reshape(-1, 4)[:, 1:])

# Each lobe is now a closed shell sitting on top of the ellipsoid. Orient 
# each body outward, then union them so the buried lobe material is removed 
# and we get one solid.
bodies = filled_trimesh.split(only_watertight=True)
print(f"closed bodies: {len(bodies)} (expected ~{lobe_count + 1})")
for b in bodies:
    #check for a bad lobe edge
    #mf.clean_from_arrays(b.vertices, b.faces)
    #make sure the normals all face outward
    trimesh.repair.fix_normals(b)  
filled_trimesh = trimesh.boolean.union(bodies, engine="manifold")
print("hail - watertight:", filled_trimesh.is_watertight,
      "is_volume:", filled_trimesh.is_volume,
      "volume:", filled_trimesh.volume,
      "bodies:", filled_trimesh.body_count)
assert filled_trimesh.is_volume, "lobe union failed"

# Visualize result
p = pv.Plotter()
p.subplot(0, 0)
p.add_mesh(filled_trimesh)
p.show()

Widget(value='<iframe src="http://localhost:55974/index.html?ui=P_0x396e16b10_2&reconnect=auto" class="pyvista…

Found 151 hole loops
Created mesh with 785358 triangles, 416922 points
Original mesh: 392832 points, 761419 cells
Filled mesh: 416922 points, 785358 cells
closed bodies: 152 (expected ~152)
hail - watertight: True is_volume: True volume: 235817.98887645596 bodies: 1


Widget(value='<iframe src="http://localhost:55974/index.html?ui=P_0x396e8e5d0_3&reconnect=auto" class="pyvista…

## Set up parameters for screw thread

In [10]:
# ======================= PARAMETERS =====================================
OUTPUT_TOP_STL     = "ellipsoid_surf_top.stl"
OUTPUT_BOTTOM_STL  = "ellipsoid_surf_bottom.stl"
 
CAVITY_CENTER      = np.array([0.0, 0.0, 0.0]) # coordinates are 0 in the center
CAVITY_RADIUS      = 18 #spherical hole for instrumentation, in mm
                        #pressure loggers are 31 x 22 x 20 mm. So set to 36 mm diameter
CUT_PLANE_NORMAL   = np.array([0.0, 0.0, 1.0]) # cutting in the vertical
CUT_PLANE_POINT    = CAVITY_CENTER
 
THREAD_SIZE        = "M40SP400"  #standard that corresponds to the diameter of the screw.
                                 #40 is the diameter here. 400 corresponds to the finish;
                                 #different finishes have different available diameters.
                                 # see thread.py in the cq_warehouse package.
THREAD_LENGTH      = 8.5 # depth (or height) of the screw. Also the size of the hole
                         # into which the screw goes. mm. 
WALL_THICKNESS     = 1.0 # width of the wall around the screw hole. mm
MANUFACTURING_COMP = 0.20 # mm print clearance - start at 0.2, tune from test prints
 
EXPORT_TOLERANCE         = 0.01 #when writing cq instance out/converting to STL file. Keep low 
EXPORT_ANGULAR_TOLERANCE = 0.05
 
ROUND_DECIMALS = 2  # for merging tessellation seam vertices after STL export

## Generate the screw (threaded collar) and receiver

In [11]:
print ('making the screw files')

plug_thread = PlasticBottleThread(size=THREAD_SIZE, external=True,
    manufacturingCompensation=MANUFACTURING_COMP)

socket_thread = PlasticBottleThread(size=THREAD_SIZE, external=False,
    manufacturingCompensation=MANUFACTURING_COMP)

# Sanity check on bottle thread design - for an external (plug) thread the root_radius
# value should be the smaller of the two. (It's the valley between the screw ridges.)
# For an internal thread (socket) root_radius should be the larger of the two (where it
# goes into the bore wall). If either assertion fails, the root/apex convention is flipped 
# from what's expected, so will need to switch it.
print("plug   root/apex/pitch:", plug_thread.root_radius, plug_thread.apex_radius, 
      plug_thread.pitch)
print("socket root/apex/pitch:", socket_thread.root_radius, socket_thread.apex_radius, 
      socket_thread.pitch)
assert plug_thread.root_radius < plug_thread.apex_radius
assert socket_thread.root_radius > socket_thread.apex_radius

# also, the "pitch" of a screw is how deep the screw had has to go in order to make one
# full turn. It needs to be able to make at least one full turn (ideally more), so check
# that thread length (screw length) is larger than the screw pitch.
assert THREAD_LENGTH > plug_thread.pitch, "THREAD_LENGTH should exceed one full pitch"

#check that the generated cq instances are valid ones so far
print("plug_thread.isValid():", plug_thread.isValid())
print("socket_thread.isValid():", socket_thread.isValid())


OVERLAP = 0.2  # mm - core intentionally overlaps INTO the thread ridge's
                # material span, so the fuse has genuine 3D overlap to merge
                # rather than two shells that only touch along a coincident
                # surface (which pymeshfix's remove_smallest_components would
                # then delete outright as a disconnected "small" component)


# Add the "core" to the screw (and the wall around the screw hole/socket) so it isn't just
# a floating spiral ridge. The screw core extends to the valleys of the screw (root_radius).
plug_core = cq.Workplane("XY").circle(plug_thread.root_radius + OVERLAP).extrude(THREAD_LENGTH)
# Socket core is a hole cut through a cylinder, where the root radius matches the valleys 
# in the socket wall.
socket_core = (cq.Workplane("XY").circle(socket_thread.root_radius + WALL_THICKNESS)
               .circle(socket_thread.root_radius - OVERLAP).extrude(THREAD_LENGTH))


#check again that the generated cq instances are valid ones so far
print("plug_core.isValid():", plug_thread.isValid())
print("socket_core.isValid():", socket_thread.isValid())


#Fuse the core/hole wall to the screw threads themselves
plug_collar_cq = plug_thread.fuse(plug_core.val())
socket_collar_cq = socket_thread.fuse(socket_core.val())
# and once again check for validity
print("plug_collar_cq.isValid():", plug_collar_cq.isValid())
print("socket_collar_cq.isValid():", socket_collar_cq.isValid())


#Some cleaning to help with validity
plug_collar_cq = plug_collar_cq.clean()
socket_collar_cq = socket_collar_cq.clean()
print("after .clean() - plug valid:", plug_collar_cq.isValid(),
      "socket valid:", socket_collar_cq.isValid())


#finally, export
cq.exporters.export(plug_collar_cq, "plug_thread_collar.stl",
                     tolerance=EXPORT_TOLERANCE, angularTolerance=EXPORT_ANGULAR_TOLERANCE)
cq.exporters.export(socket_collar_cq, "socket_thread_collar.stl",
                     tolerance=EXPORT_TOLERANCE, angularTolerance=EXPORT_ANGULAR_TOLERANCE)
print("exported plug_thread_collar.stl and socket_thread_collar.stl")



making the screw files
plug   root/apex/pitch: 18.294999999999998 19.485 4.233333333333333
socket root/apex/pitch: 20.265 19.075 4.233333333333333
plug_thread.isValid(): True
socket_thread.isValid(): True
plug_core.isValid(): True
socket_core.isValid(): True
plug_collar_cq.isValid(): True
socket_collar_cq.isValid(): True
after .clean() - plug valid: True socket valid: True
exported plug_thread_collar.stl and socket_thread_collar.stl


## Finish conversion of CQ screw threaded objects to trimesh meshes

In [12]:
print ('finish converting cq instances to trimesh')

def cq_to_trimesh(cq_solid, tol, ang_tol):
    verts, faces = cq_solid.tessellate(tol, ang_tol)
    v = np.array([(p.x, p.y, p.z) for p in verts], dtype=np.float64)
    f = np.array(faces, dtype=np.int64)
    return trimesh.Trimesh(vertices=v, faces=f, process=True)
 
plug_collar = cq_to_trimesh(plug_collar_cq, EXPORT_TOLERANCE, EXPORT_ANGULAR_TOLERANCE)
socket_collar = cq_to_trimesh(socket_collar_cq, EXPORT_TOLERANCE, EXPORT_ANGULAR_TOLERANCE)

print (socket_collar.bounds)
print (socket_thread.root_radius + WALL_THICKNESS)
print (plug_collar.bounds)
print (THREAD_LENGTH)

#check if it's a valid volume now
for name, collar in [("plug_collar", plug_collar), ("socket_collar", socket_collar)]:
    print(f"{name} raw - watertight: {collar.is_watertight}, is_volume: {collar.is_volume}")
    



#plot them just to see
p = pv.Plotter(shape=(1,2))
p.subplot(0, 0)
p.add_mesh(socket_collar)

p.subplot(0, 1)
p.add_mesh(plug_collar)

p.show()

finish converting cq instances to trimesh
[[-21.265 -21.265   0.   ]
 [ 21.265  21.265   8.5  ]]
21.265
[[-19.48379045 -19.48500015   0.        ]
 [ 19.48500015  19.48469764   8.5       ]]
8.5
plug_collar raw - watertight: True, is_volume: True
socket_collar raw - watertight: True, is_volume: True


Widget(value='<iframe src="http://localhost:55974/index.html?ui=P_0x395df4f50_4&reconnect=auto" class="pyvista…

## Do the hailstone slicing

In [21]:
print ('slicing hailstone in half')
top = trimesh.intersections.slice_mesh_plane(filled_trimesh, plane_normal=CUT_PLANE_NORMAL, 
                                             plane_origin=CUT_PLANE_POINT, cap=True)

bottom = trimesh.intersections.slice_mesh_plane(filled_trimesh, plane_normal=-CUT_PLANE_NORMAL, 
                                                plane_origin=CUT_PLANE_POINT, cap=True)

#make sure these meshes are both valid
print("top   - watertight:", top.is_watertight, "is_volume:", top.is_volume, "volume:", top.volume)
print("bottom- watertight:", bottom.is_watertight, "is_volume:", bottom.is_volume, "volume:", bottom.volume)
assert top.is_volume and bottom.is_volume, "slicing did not produce valid capped volumes"
print("total hailstone volume:", filled_trimesh.volume)

# remove a cylindrical hole the size of socket_thread.root_radius + WALL_THICKNESS - OVERLAP from top
# The socket_thread.root_radius + WALL_THICKNESS is how big socket_collar is, the -OVERLAP is to ensure the objects overlap
cavity = trimesh.creation.cylinder(subdivisions=4, radius=socket_thread.root_radius + 0.2*WALL_THICKNESS - OVERLAP,
                                   height = THREAD_LENGTH+OVERLAP)
cavity.apply_translation(np.array([0.0, 0.0, (THREAD_LENGTH+OVERLAP)/2.]))

print (socket_thread.root_radius, WALL_THICKNESS, OVERLAP)
 
top = trimesh.boolean.difference([top, cavity], engine="manifold")
print("top - cavity is watertight:", top.is_watertight, "is_volume:", 
      top.is_volume, "volume:", top.volume)


#make a plot to be certain it worked
p = pv.Plotter(shape=(1,2))
p.subplot(0, 0)
p.add_mesh(top, opacity=0.8, color='light_blue')
p.add_mesh(socket_collar, color='red')

p.subplot(0, 1)
p.add_mesh(bottom, opacity=0.3, color='light_blue')
p.add_mesh(plug_collar, color='red')

p.show()

slicing hailstone in half
top   - watertight: True is_volume: True volume: 117713.38782298651
bottom- watertight: True is_volume: True volume: 118104.60105346947
total hailstone volume: 235817.98887645596
20.265 1.0 0.2
top - cavity is watertight: True is_volume: True volume: 106560.9923702912


Widget(value='<iframe src="http://localhost:55974/index.html?ui=P_0x3ca85b910_11&reconnect=auto" class="pyvist…

## Set the thread phase so the two halves screw together exactly at a known longitude

In [14]:
print('setting the thread phase so the halves seat at a known longitude')
# At full seating the faces touch at z=0, so the socket sits where it was modelled except
# for an unknown rotation phi about z. Sweep phi, measure plug/socket collision at each
# angle, and find the band of angles where they fit. Tightening a right-hand thread turns
# the top CLOCKWISE seen from above (decreasing phi), so it stops at the band's low edge.
from trimesh.transformations import rotation_matrix

def socket_rotated(phi_deg):
    #rotate and return a copy of the socket, rotated phi_deg about z axis.
    s = socket_collar.copy()
    s.apply_transform(rotation_matrix(np.radians(phi_deg), [0.0, 0.0, 1.0]))
    return s

def interference(phi_deg):
    #checks to see if the plug and rotated socket threads overlap in volume, and returns how much.
    #hint: if they overlap, this angle doesn't work.
    x = trimesh.boolean.intersection([plug_collar, socket_rotated(phi_deg)], engine="manifold")
    return 0.0 if len(x.faces) == 0 else float(x.volume)

#Loop through the whole range of possible rotation angles, and calculate volume overlap.
PHASE_STEP_DEG = 5.0
phis = np.arange(0.0, 360.0, PHASE_STEP_DEG)
vols = np.array([interference(p) for p in phis])

#Once we know the max and min volume overlap per angle, we can create a threshold that is 
# just slightly larger than the min volume overlap measured.
v_lo, v_hi = vols.min(), vols.max()
thresh = v_lo + 0.02 * (v_hi - v_lo)      # tolerant of any constant, angle-independent overlap
print(f"interference over one turn: min {v_lo:.2f}, max {v_hi:.2f} mm^3")
#make sure the threads actually interact
assert v_hi - v_lo > 1.0, "interference barely varies with angle - check both collars span z=0..THREAD_LENGTH"

#Now find which angles actually work
free = vols <= thresh
assert free.any() and not free.all(), "no clear free band - try a smaller PHASE_STEP_DEG"

#The angle we want will be surrounded by one phi that works and one that doesn't.
# In case these fall across the 360-0 phi angle border, do some array shifting.
shift  = int(np.argmin(free)) #first angle that doesn't work
free_r = np.roll(free, -shift)  
#calculate the angles that match this shifted array (extend to 365, etc.)
phis_u = phis[shift] + PHASE_STEP_DEG * np.arange(len(phis))   # unwrapped angles

#Find the longest stretch of fitting angles (i.e., "Trues"). 
# The very first angle of this stretch is the first angle that fits. We want that one.
# Why this edge (the start of the fitting range, the low-angle side) and not the other one?
# Increasing φ means turning counterclockwise when seen from above. Walking up the array, 
# i0 is where the fitting range begins, which makes it the range’s clockwise-most end. 
# You tighten a right-hand thread by turning the top clockwise, so once the faces touch, 
# more tightening pushes the top clockwise until a thread flank hits. That’s the clockwise 
# end of the fitting range.
runs, i = [], 0
while i < len(free_r):
    if free_r[i]:
        j = i
        while j < len(free_r) and free_r[j]:
            j += 1
        runs.append((i, j))
        i = j
    else:
        i += 1
if len(runs) > 1:
    print(f"WARNING: {len(runs)} free bands found - using the widest")

#i0 and i1 are the first angle that fits (i1) and the angle before it (i0) that collides.
i0, i1 = max(runs, key=lambda r: r[1] - r[0])
#get the phi values that correspond to i0 and i1
a, b = phis_u[i0 - 1], phis_u[i0]

#narrow down the phi inside these two angles.
for _ in range(6):
    m = 0.5 * (a + b)
    a, b = (a, m) if interference(m) <= thresh else (m, b)
seat_edge  = b

#Print out a diagnostic showing the number of angles that fit (times 5 degrees)
band_width = (i1 - i0) * PHASE_STEP_DEG
print(f"seats at phi = {seat_edge % 360:.1f} deg; free play when loose ~{band_width:.0f} deg "
      f"(~{np.radians(band_width) * min(radius_a, radius_c):.1f} mm at the equator)")

#Add in a correction identified from a test print
PHASE_TRIM_DEG = np.degrees(3 / radius_a)      # empirical correction from a test print - see below
#Given our calculations, how much rotation should we build into the socket?
SEAT_PHASE_DEG = (seat_edge + PHASE_TRIM_DEG) % 360.0
socket_collar_phased = socket_rotated(SEAT_PHASE_DEG)
print(f"socket collar rotated by {SEAT_PHASE_DEG:.1f} deg")

setting the thread phase so the halves seat at a known longitude
interference over one turn: min 0.00, max 34.22 mm^3
seats at phi = 169.8 deg; free play when loose ~160 deg (~111.7 mm at the equator)
socket collar rotated by 174.1 deg


## Attach the screw and threads

In [15]:
print ('attaching screw and socket')
plug_collar.apply_translation(CAVITY_CENTER)
socket_collar_phased.apply_translation(CAVITY_CENTER)
 
bottom = trimesh.boolean.union([bottom, plug_collar], engine="manifold")
print("bottom + plug, watertight:", bottom.is_watertight, "is_volume:", bottom.is_volume)
 
top = trimesh.boolean.union([top, socket_collar_phased], engine="manifold")
print("top + socket collar, watertight:", top.is_watertight, "is_volume:", top.is_volume)
 
assert top.is_volume and bottom.is_volume, "collar attach step broke watertightness - stop here and reconsider your life choices"


#make a plot to be certain it worked
p = pv.Plotter(shape=(1,2))
p.subplot(0, 0)
p.add_mesh(top)

p.subplot(0, 1)
p.add_mesh(bottom)

p.show()


attaching screw and socket
bottom + plug, watertight: True is_volume: True
top + socket collar, watertight: True is_volume: True


Widget(value='<iframe src="http://localhost:55974/index.html?ui=P_0x395e1d910_6&reconnect=auto" class="pyvista…

## Add a registration rabbet (fixes concentricity)

A **Rabbet** (a raised ring on `bottom` nesting into a matching groove on `top`) - forces true concentric alignment. 

In [16]:
print('adding registration rabbet + helical key')

REG_HEIGHT    = 1.0   # axial height of the rabbet - shallow, only needs to register
REG_WIDTH     = 2.0   # radial width of the rabbet ring, mm
REG_CLEARANCE = 0.2   # radial clearance on the groove so the boss actually fits
SAFE_SKIN     = 1.0   # mm of solid shell to leave between the outermost feature and
                      # the hailstone surface. Raise this if a feature still breaks out.
                      
# ---- analytic bound on the available land ----
# Use the base spheroid, which is a hard lower bound on the real surface 
# (lobes only add material outside it).
#
# A spheroid's cross-section shrinks with |z|, so the thinnest skin - and therefore the
# binding constraint - is at the HIGHEST z that any registration feature reaches.
def spheroid_radius_at_z(z):
    """Smallest in-plane radius of the base spheroid at height z. 0 above the pole."""
    f = 1.0 - (z / radius_b) ** 2
    return 0.0 if f <= 0.0 else min(radius_a, radius_c) * np.sqrt(f)

collar_edge = socket_thread.root_radius + WALL_THICKNESS

# Find the horizontal (radial) extents of the rabbet.
# Outer edge: half the rabbet width, plus groove clearance, plus the skin to keep.
# Inner limit: how far in the rabbet can go without sticking into the screw. 
rabbet_outer_edge = REG_WIDTH / 2.0 + REG_CLEARANCE + SAFE_SKIN
rabbet_inner_limit   = collar_edge + REG_WIDTH / 2.0

#how far away from the z cut plane will either the screw threads or rabbet extend?
z_rabbet_top = REG_HEIGHT + OVERLAP
z_screw_top = THREAD_LENGTH + OVERLAP
Z_FEATURE_MAX    = max(z_rabbet_top, z_screw_top)   # whichever climbs higher governs
available_radius = spheroid_radius_at_z(Z_FEATURE_MAX)
r_outer_limit    = available_radius - rabbet_outer_edge

# how big can the rabbet be?
REG_RADIUS = 0.5 * (rabbet_inner_limit + r_outer_limit)   # centered in the legal band
print(f"rabbet ring centerline at: {REG_RADIUS:.2f} mm "
      f"(legal band {rabbet_inner_limit:.2f} - {r_outer_limit:.2f} mm)")


# ---- rabbet boss + groove (concentricity) ----
# Both extend slightly below z=0 (by OVERLAP) into bottom's existing material for a
# genuine fuse overlap, same reasoning as everywhere else in this notebook.
rabbet_boss = trimesh.creation.annulus(
    r_min=REG_RADIUS - REG_WIDTH/2, r_max=REG_RADIUS + REG_WIDTH/2,
    height=REG_HEIGHT + OVERLAP, sections=128)
rabbet_boss.apply_translation(CAVITY_CENTER + np.array([0.0, 0.0, (REG_HEIGHT - OVERLAP)/2.0]))

groove_pocket = trimesh.creation.annulus(
    r_min=REG_RADIUS - REG_WIDTH/2 - REG_CLEARANCE,
    r_max=REG_RADIUS + REG_WIDTH/2 + REG_CLEARANCE,
    height=REG_HEIGHT + OVERLAP, sections=128)
groove_pocket.apply_translation(CAVITY_CENTER + np.array([0.0, 0.0, (REG_HEIGHT + OVERLAP)/2.0]))

# ---- attach everything ----
bottom = trimesh.boolean.union([bottom, rabbet_boss], engine="manifold")
print("bottom + rabbet, watertight:", bottom.is_watertight, "is_volume:", bottom.is_volume)

top = trimesh.boolean.difference([top, groove_pocket], engine="manifold")
print("top - groove, watertight:", top.is_watertight, "is_volume:", top.is_volume)

assert top.is_volume and bottom.is_volume, "registration feature broke watertightness - stop here"

#make a plot to check the rabbet + key look right
p = pv.Plotter(shape=(1,2))
p.subplot(0, 0)
p.add_mesh(top, opacity=0.8)
p.subplot(0, 1)
p.add_mesh(bottom, opacity=0.8)
p.show()

adding registration rabbet + helical key
rabbet ring centerline at: 29.17 mm (legal band 22.27 - 36.08 mm)
bottom + rabbet, watertight: True is_volume: True
top - groove, watertight: True is_volume: True


Widget(value='<iframe src="http://localhost:55974/index.html?ui=P_0x396e43d10_7&reconnect=auto" class="pyvista…

## Remove the spherical hole for instrumentation from each half

In [17]:
print ('making instrumentation hole')
cavity = trimesh.creation.icosphere(subdivisions=4, radius=CAVITY_RADIUS)
cavity.apply_translation(CAVITY_CENTER)
 
top = trimesh.boolean.difference([top, cavity], engine="manifold")
print("top - cavity is watertight:", top.is_watertight, "is_volume:", 
      top.is_volume, "volume:", top.volume)
 
bottom = trimesh.boolean.difference([bottom, cavity], engine="manifold")
print("bottom- cavity is watertight:", bottom.is_watertight, "is_volume:", 
      bottom.is_volume, "volume:", bottom.volume)

#make a plot to be certain it worked
p = pv.Plotter(shape=(1,2))
p.subplot(0, 0)
p.add_mesh(top)

p.subplot(0, 1)
p.add_mesh(bottom)

p.show()

print("socket root/apex:", socket_thread.root_radius, socket_thread.apex_radius)
print("CAVITY_RADIUS:", CAVITY_RADIUS)

making instrumentation hole
top - cavity is watertight: True is_volume: True volume: 102312.99376433216
bottom- cavity is watertight: True is_volume: True volume: 107582.40118154282


Widget(value='<iframe src="http://localhost:55974/index.html?ui=P_0x396709810_8&reconnect=auto" class="pyvista…

socket root/apex: 20.265 19.075
CAVITY_RADIUS: 18


## Finally, write the blasted things out

In [18]:
print ('writing out')
for name, half, path in [("top", top, OUTPUT_TOP_STL), ("bottom", bottom, OUTPUT_BOTTOM_STL)]:
    if not half.is_watertight:
        print ('fixing ', name)
        v, f = mf.clean_from_arrays(half.vertices, half.faces)
        half = trimesh.Trimesh(v, f)
    half.export(path)
    print(f"{path}: watertight={half.is_watertight}, volume={half.volume:.1f} mm^3")

writing out
ellipsoid_surf_top.stl: watertight=True, volume=102313.0 mm^3
ellipsoid_surf_bottom.stl: watertight=True, volume=107582.4 mm^3


## Debugging plots

In [19]:
top_check = pv.wrap(top)  # or pv.read(OUTPUT_TOP_STL) to check the actual exported file
clipped = top_check.clip(normal='x', origin=(0, 0, 0))

plotter = pv.Plotter()
plotter.add_mesh(clipped, color='lightblue', show_edges=True)
plotter.add_mesh(clipped.extract_feature_edges(), color='red', line_width=2)
plotter.show()

Widget(value='<iframe src="http://localhost:55974/index.html?ui=P_0x396f260d0_9&reconnect=auto" class="pyvista…

In [20]:
bottom_check = pv.wrap(bottom)
clipped_b = bottom_check.clip(normal='x', origin=(0, 0, 0))
plotter = pv.Plotter()
plotter.add_mesh(clipped_b, color='lightgreen', show_edges=True)
plotter.show()

overlap = trimesh.boolean.intersection([plug_collar, socket_collar], engine="manifold")
print("interference volume:", overlap.volume if overlap is not None else 0)


Widget(value='<iframe src="http://localhost:55974/index.html?ui=P_0x353c51790_10&reconnect=auto" class="pyvist…

interference volume: 16.03266046743177
